In [ ]:
!git clone https://github.com/nasoskar/NLP.git
%cd NLP

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.makedirs("checkpoints", exist_ok=True)

Tweets dataset already to integers:
0 -> bearish (negative)
1 -> bullish (positive)
2 -> neutral 

In [ ]:
import pandas as pd

phrasebank_df = pd.read_csv(
    '/content/drive/MyDrive/NLP/FinancialPhraseBank-v1.0/Sentences_75Agree.txt',
    sep='@',
    header=None,
    names=['text', 'sentiment'],
    encoding='latin-1'  # needed for special characters
)

label_map = {"neutral": 2, "positive": 1, "negative": 0}
phrasebank_df['sentiment'] = phrasebank_df['sentiment'].map(label_map)

phrasebank_df.rename(columns={"sentiment": "label"}, inplace=True)

splits = {'train': 'sent_train.csv', 'validation': 'sent_valid.csv'}
df_tr = pd.read_csv("hf://datasets/zeroshot/twitter-financial-news-sentiment/" + splits["train"])
df_val = pd.read_csv("hf://datasets/zeroshot/twitter-financial-news-sentiment/" + splits["validation"])
twitter_df = pd.concat([df_tr, df_val], ignore_index=True)

total_df = pd.concat([phrasebank_df, twitter_df], ignore_index=True, sort=False)

import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)   # remove URLs
    text = re.sub(r'\$[a-zA-Z]+', 'TICKER', text) # normalize tickers
    return text

import pandas as pd

pd.set_option('display.max_colwidth', None)

total_df["text"] = total_df["text"].apply(clean_text)


In [3]:
# Class distribution
print(total_df["label"].value_counts(normalize=True))

# Word count stats per bucket
total_df["word_count"] = total_df["text"].str.split().str.len()
print(total_df["word_count"].describe())

# Length bucket distribution
def assign_bucket(wc):
    if wc <= 8: return "short"
    elif wc <= 20: return "medium"
    else: return "long"

total_df["bucket"] = total_df["word_count"].apply(assign_bucket)
print(total_df["bucket"].value_counts())

label
2    0.642876
1    0.213534
0    0.143591
Name: proportion, dtype: float64
count    15384.000000
mean        14.158736
std          7.700136
min          0.000000
25%          9.000000
50%         12.000000
75%         17.000000
max         81.000000
Name: word_count, dtype: float64
bucket
medium    10148
short      3028
long       2208
Name: count, dtype: int64


In [4]:
# See the distribution
print(total_df["word_count"].quantile([0.90, 0.95, 0.99]))

0.90    23.0
0.95    30.0
0.99    43.0
Name: word_count, dtype: float64


In [5]:
from dataset import create_dataframe, split_dataset, clean_text
from svm import SVM_Classifier

In [6]:
#Step 1: create dataframe based on the image paths
df = create_dataframe()
df['text'] = df['text'].apply(clean_text)
print(df)

                                                                                                                                                                                                                 text  \
0                                                                                     according to gran , the company has no plans to move all production to russia , although that is where the company is growing .   
1      with the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .   
2                   for the last quarter of 2010 , componenta 's net sales doubled to eur131m from eur76m for the same period a year earlier , while it moved to a zero pre-tax profit from a pre-tax loss of eur7m .   
3                                                                                       in the third quarter of 2010 , net sales inc

In [7]:
#Step 2: Split the dataset
X_train, y_train, X_val, y_val, X_test, y_test = split_dataset(df)


In [4]:
#Step 3: SVM with TF-IDF
svmclass = SVM_Classifier()
svmclass.fit(X_train, y_train)
y_pred = svmclass.predict(X_test)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best params: {'C': 1, 'kernel': 'rbf'}
Best macro F1: 0.7485001971647353


In [5]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Overall metrics
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Macro F1:", round(f1_score(y_test, y_pred, average="macro"), 3))

# Per class breakdown
print(classification_report(y_test, y_pred, target_names=["negative", "positive", "neutral"]))

Accuracy: 0.834
Macro F1: 0.766
              precision    recall  f1-score   support

    negative       0.76      0.58      0.66       221
    positive       0.75      0.74      0.74       329
     neutral       0.87      0.92      0.90       989

    accuracy                           0.83      1539
   macro avg       0.79      0.75      0.77      1539
weighted avg       0.83      0.83      0.83      1539



In [6]:
from evaluate import (
    evaluate_overall,
    evaluate_by_bucket, 
    plot_confusion_matrix,
    build_results_table,
    plot_bucket_comparison
)

# After each model
overall = evaluate_overall(y_test, y_pred, model_name="SVM")
buckets = evaluate_by_bucket(y_test, y_pred, df["bucket"], model_name="SVM")
plot_confusion_matrix(y_test, y_pred, model_name="SVM")


SVM — Overall Results
Accuracy:  0.834
Macro F1:  0.766

              precision    recall  f1-score   support

    negative       0.76      0.58      0.66       221
    positive       0.75      0.74      0.74       329
     neutral       0.87      0.92      0.90       989

    accuracy                           0.83      1539
   macro avg       0.79      0.75      0.77      1539
weighted avg       0.83      0.83      0.83      1539



KeyError: 'bucket'

In [8]:
from dataset import build_vocab, LSTMClassification

vocab = build_vocab(X_train)

train_dataset = LSTMClassification(X_train, y_train, vocab)
val_dataset   = LSTMClassification(X_val,   y_val,   vocab)  
test_dataset  = LSTMClassification(X_test,  y_test,  vocab)  

Vocabulary size: 26519


In [ ]:
from torch.utils.data import DataLoader
from config import *
#Step 4:  Call DataLoader
train_dataloader = DataLoader(train_dataset, batch_size=LSTM_BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_dataloader = DataLoader(val_dataset, batch_size=LSTM_BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

batch = next(iter(train_dataloader))
input_ids = batch["input_ids"]  # shape: (32, 30)
labels    = batch["label"]      # shape: (32,)

In [10]:
import torch
from train import load_glove, compute_class_weight, LSTM_train, LSTM
import numpy as np
from config import *
from lstm import *
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
from itertools import product

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

best_f1 = 0
best_params = {}

for hidden_size, dropout, lr in product(HIDDEN_SIZES, DROPOUT_RATES, LEARNING_RATES):
    print(f"\nTrying hidden={hidden_size}, dropout={dropout}, lr={lr}")

    # Step 5: Call LSTM model
    glove_matrix = load_glove("glove.6B.100d.txt", vocab)
    weights = compute_class_weight("balanced", classes=np.array([0,1,2]), y=y_train)
    class_weights = torch.tensor(weights, dtype=torch.float)
    model = LSTM(len(vocab), 100, hidden_size, 3, dropout).to(device)
    model.embedding.weight.data.copy_(glove_matrix)
    model.embedding.weight.requires_grad = True

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    val_f1 = LSTM_train(model, optimizer, criterion, train_dataloader, val_dataloader)

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_params = {"hidden_size": hidden_size, "dropout": dropout, "lr": lr}
        torch.save(model.state_dict(), "checkpoints/best_lstm_overall.pth")

print(f"\nBest params: {best_params}")
print(f"Best val F1: {best_f1:.4f}")


Trying hidden=64, dropout=0, lr=0.001
Epoch 1/20 | Train Loss: 0.9897 | Val Loss: 0.8916 | Val Macro F1: 0.4712
Epoch 2/20 | Train Loss: 0.7937 | Val Loss: 0.8384 | Val Macro F1: 0.4689
Epoch 3/20 | Train Loss: 0.6571 | Val Loss: 0.8499 | Val Macro F1: 0.4696
Epoch 4/20 | Train Loss: 0.5308 | Val Loss: 0.9466 | Val Macro F1: 0.5874
Epoch 5/20 | Train Loss: 0.3608 | Val Loss: 0.8293 | Val Macro F1: 0.6701
Epoch 6/20 | Train Loss: 0.2176 | Val Loss: 0.9263 | Val Macro F1: 0.7061
Epoch 7/20 | Train Loss: 0.1363 | Val Loss: 0.8759 | Val Macro F1: 0.6964
Epoch 8/20 | Train Loss: 0.0890 | Val Loss: 1.1915 | Val Macro F1: 0.6844
Epoch 9/20 | Train Loss: 0.0674 | Val Loss: 1.1296 | Val Macro F1: 0.7111
Early stopping at epoch 9

Trying hidden=64, dropout=0, lr=0.0001
Epoch 1/20 | Train Loss: 1.0982 | Val Loss: 1.0948 | Val Macro F1: 0.3138
Epoch 2/20 | Train Loss: 1.0919 | Val Loss: 1.0893 | Val Macro F1: 0.3270
Epoch 3/20 | Train Loss: 1.0241 | Val Loss: 0.9335 | Val Macro F1: 0.4417
Epoch 4

RuntimeError: cuDNN error: CUDNN_STATUS_EXECUTION_FAILED_CUDART

In [ ]:
import torch
from finbert import FINBERT
from train import finbert_train
from config import *
from lstm import *
from itertools import product

#Step 4:  Call DataLoader
train_dataloader = DataLoader(train_dataset, batch_size=FINBERT_BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_dataloader = DataLoader(val_dataset, batch_size=FINBERT_BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

batch = next(iter(train_dataloader))
input_ids = batch["input_ids"]  # shape: (32, 30)
labels    = batch["label"]      # shape: (32,)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

best_f1 = 0
best_params = {}

for dropout, lr in product(FINBERT_DROPOUTS, FINBERT_LRS):
    print(f"\nTrying dropout={dropout}, lr={lr}")

    # Step 5: Call FINBERT model
    model = FINBERT(dropout).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    val_f1 = finbert_train(model, optimizer, criterion, train_dataloader, val_dataloader)

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_params = {"dropout": dropout, "lr": lr}
        torch.save(model.state_dict(), "checkpoints/best_finbert_overall.pth")

print(f"\nBest params: {best_params}")
print(f"Best val F1: {best_f1:.4f}")